<a href="https://colab.research.google.com/github/Sapikzzz/AI-ASL/blob/main/ViT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import numpy as np
import os
import cv2
from sklearn.utils import shuffle
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import gc
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D, Activation
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as preprocess_mobilenetv2
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import BatchNormalization
import kagglehub
import seaborn as sns
import pandas as pd
import skimage
from skimage.transform import resize
from sklearn.metrics import classification_report, confusion_matrix
import os
from glob import glob
import random
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.layers import Input, Layer, MultiHeadAttention, LayerNormalization, Dense
from tensorflow.keras.layers import Add, Embedding, Reshape
from tensorflow.keras.initializers import glorot_uniform
import math

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

In [16]:
import kagglehub
grassknoted_asl_alphabet_path = kagglehub.dataset_download('grassknoted/asl-alphabet')

train_path = os.path.join(grassknoted_asl_alphabet_path, '/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train')
print('Data source import complete.')

Data source import complete.


In [17]:
print("Num GPUs Available: ", len(tf.config.list_physical_devices('GPU')))
tf.config.list_physical_devices()

tf.test.is_gpu_available()

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
            print("GPU configured successfully")
    except RuntimeError as e:
        print("Error configuring GPU:", e)

Num GPUs Available:  1
GPU configured successfully


In [18]:
target_size = (96, 96)
target_dims = (96, 96, 3)
n_classes = 29
val_frac = 0.1
batch_size = 256

data_augmentor = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=False,
    rescale=1./255,
    validation_split=val_frac
)

In [19]:
train_gen = data_augmentor.flow_from_directory(train_path, target_size=target_size, batch_size=batch_size, shuffle=True, subset='training', class_mode='sparse')
val_gen = data_augmentor.flow_from_directory(train_path, target_size=target_size, batch_size=batch_size, subset='validation', class_mode='sparse')

Found 78300 images belonging to 29 classes.
Found 8700 images belonging to 29 classes.


In [23]:
import tensorflow as tf
from tensorflow.keras import layers, models, initializers

class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size, projection_dim):
        super().__init__()
        self.patch_size = patch_size
        self.projection_dim = projection_dim
        self.projection = layers.Conv2D(
            filters=projection_dim,
            kernel_size=patch_size,
            strides=patch_size,
            padding='valid'
        )

    def call(self, images):
        # Shape: (batch_size, height/patch, width/patch, projection_dim)
        patches = self.projection(images)
        # Flatten to (batch_size, num_patches, projection_dim)
        patch_dims = tf.shape(patches)[1] * tf.shape(patches)[2]
        return tf.reshape(patches, (tf.shape(images)[0], patch_dims, self.projection_dim))

class AddPositionEmbedding(layers.Layer):
    def __init__(self, num_patches, projection_dim):
        super().__init__()
        self.position_embedding = self.add_weight(
            name="pos_embed", shape=(1, num_patches + 1, projection_dim),
            initializer=initializers.RandomNormal(stddev=0.06)
        )
        self.class_token = self.add_weight(
            name="cls_token", shape=(1, 1, projection_dim),
            initializer=initializers.Zeros()
        )

    def call(self, x):
        batch_size = tf.shape(x)[0]
        class_token = tf.broadcast_to(self.class_token, [batch_size, 1, tf.shape(x)[-1]])
        x = tf.concat([class_token, x], axis=1)
        return x + self.position_embedding

def transformer_encoder(x, num_heads, ff_dim, dropout_rate):
    # Layer normalization and multi-head attention
    attn_input = layers.LayerNormalization(epsilon=1e-6)(x)
    attention_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=ff_dim)(attn_input, attn_input)
    attention_output = layers.Dropout(dropout_rate)(attention_output)
    out1 = layers.Add()([x, attention_output])

    # Layer normalization and MLP
    ffn_input = layers.LayerNormalization(epsilon=1e-6)(out1)
    ffn_output = layers.Dense(ff_dim * 2, activation='gelu')(ffn_input)
    ffn_output = layers.Dense(ff_dim)(ffn_output)
    ffn_output = layers.Dropout(dropout_rate)(ffn_output)
    return layers.Add()([out1, ffn_output])

def build_vit_model(
    input_shape=(96, 96, 3),
    patch_size=(16, 16),
    num_layers=6,
    projection_dim=64,
    num_heads=4,
    ff_dropout=0.1,
    num_classes=29
):
    inputs = layers.Input(shape=input_shape)

    # Compute number of patches
    num_patches = (input_shape[0] // patch_size[0]) * (input_shape[1] // patch_size[1])

    # Patch embedding
    x = PatchEmbedding(patch_size, projection_dim)(inputs)
    x = AddPositionEmbedding(num_patches, projection_dim)(x)

    # Transformer layers
    for _ in range(num_layers):
        x = transformer_encoder(x, num_heads, projection_dim, ff_dropout)

    # Classification head using class token (first token)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    class_token = x[:, 0]  # Extract class token
    x = layers.Dropout(ff_dropout)(class_token)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return models.Model(inputs, outputs)

vit_model = build_vit_model()

vit_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False),
    metrics=['accuracy']
)
vit_model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 96, 96, 3) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_embedding_3   │ (None, 36, 64)    │     49,216 │ input_layer_3[0]… │
│ (PatchEmbedding)    │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_position_embed… │ (None, 37, 64)    │      2,432 │ patch_embedding_… │
│ (AddPositionEmbedd… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 37, 64)    │        128 │ add_position_emb… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 37, 64)    │     66,368 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_58          │ (None, 37, 64)    │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_36 (Add)        │ (None, 37, 64)    │          0 │ add_position_emb… │
│                     │                   │            │ dropout_58[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 37, 64)    │        128 │ add_36[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_39 (Dense)    │ (None, 37, 128)   │      8,320 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_40 (Dense)    │ (None, 37, 64)    │      8,256 │ dense_39[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_59          │ (None, 37, 64)    │          0 │ dense_40[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_37 (Add)        │ (None, 37, 64)    │          0 │ add_36[0][0],     │
│                     │                   │            │ dropout_59[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 37, 64)    │        128 │ add_37[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 37, 64)    │     66,368 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_61          │ (None, 37, 64)    │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_38 (Add)        │ (None, 37, 64)    │          0 │ add_37[0][0],     │
│                     │                   │            │ dropout_61[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 37, 64)    │        128 │ add_38[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 552,861 (2.11 MB)

 Trainable params: 552,861 (2.11 MB)

 Non-trainable params: 0 (0.00 B)

In [24]:
history = vit_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15
)

Epoch 1/15
306/306 ━━━━━━━━━━━━━━━━━━━━ 347s 1s/step - accuracy: 0.0411 - loss: 3.4748 - val_accuracy: 0.0551 - val_loss: 3.2523
Epoch 2/15
306/306 ━━━━━━━━━━━━━━━━━━━━ 294s 962ms/step - accuracy: 0.1250 - loss: 3.0376 - val_accuracy: 0.2403 - val_loss: 2.4392
Epoch 3/15
306/306 ━━━━━━━━━━━━━━━━━━━━ 278s 909ms/step - accuracy: 0.3732 - loss: 1.9325 - val_accuracy: 0.4632 - val_loss: 1.6059
Epoch 4/15
306/306 ━━━━━━━━━━━━━━━━━━━━ 283s 924ms/step - accuracy: 0.5699 - loss: 1.2854 - val_accuracy: 0.4952 - val_loss: 1.5075
Epoch 5/15
306/306 ━━━━━━━━━━━━━━━━━━━━ 283s 926ms/step - accuracy: 0.6765 - loss: 0.9377 - val_accuracy: 0.5936 - val_loss: 1.1983
Epoch 6/15
306/306 ━━━━━━━━━━━━━━━━━━━━ 286s 935ms/step - accuracy: 0.7487 - loss: 0.7315 - val_accuracy: 0.6333 - val_loss: 1.1140
Epoch 7/15
306/306 ━━━━━━━━━━━━━━━━━━━━ 282s 921ms/step - accuracy: 0.7963 - loss: 0.5919 - val_accuracy: 0.6964 - val_loss: 0.9606
Epoch 8/15
306/306 ━━━━━━━━━━━━━━━━━━━━ 298s 973ms/step - accuracy: 0.8369 - lo

In [22]:
vit_model.save('ViT_model.keras')